## Host CPU subprocess runner

Runs every `*.py` script under `host/` (Intel/desktop Python baselines) and saves JSON under `results/` at the **repository root**.

If execution fails to find `host/`, set your Jupyter current working directory to the repo root, or `cd` there before running the cells.

In [5]:
import json
import subprocess
import sys
import time
from pathlib import Path


def _repo_root() -> Path:
    p = Path.cwd().resolve()
    for _ in range(8):
        if (p / "host").is_dir():
            return p
        if p.parent == p:
            break
        p = p.parent
    raise FileNotFoundError(
        "Could not find project root (folder containing 'host/'). "
        "cd to the repository root or open Jupyter with that folder as cwd."
    )


REPO_ROOT = _repo_root()
CPU_DIR = REPO_ROOT / "host"
RESULTS_DIR = REPO_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

for script_path in sorted(CPU_DIR.glob("*.py")):
    name = script_path.stem
    start = time.perf_counter()
    completed = subprocess.run(
        [sys.executable, str(script_path)],
        text=True,
        capture_output=True,
    )
    elapsed = time.perf_counter() - start

    result = {
        "name": name,
        "script": str(script_path),
        "return_code": completed.returncode,
        "execution_time_s": elapsed,
        "stdout": completed.stdout.strip(),
        "stderr": completed.stderr.strip(),
    }

    print(f"[{name}] return_code={completed.returncode} time={elapsed:.6f}s")
    if completed.stdout:
        print(completed.stdout.strip())
    if completed.stderr:
        print(completed.stderr.strip())

    output_path = RESULTS_DIR / f"{name}_result.json"
    with output_path.open("w", encoding="utf-8") as f:
        json.dump({name: result}, f, indent=4)

    print(f"Saved {name} result to {output_path}")



[hello] return_code=0 time=0.085667s
Hello World
Saved hello result to results\hello_result.json
[matmul] return_code=0 time=0.172640s
shape=(32, 64) checksum=32029
Saved matmul result to results\matmul_result.json


## Load saved results

In [6]:
from pathlib import Path
import json


def _repo_root() -> Path:
    p = Path.cwd().resolve()
    for _ in range(8):
        if (p / "host").is_dir():
            return p
        if p.parent == p:
            break
        p = p.parent
    raise FileNotFoundError("Run from repo root (folder containing host/).")


REPO_ROOT = _repo_root()
RESULTS_DIR = REPO_ROOT / "results"

result_files = sorted(
    path for path in RESULTS_DIR.glob("*.json")
    if path.name != "cpu_results.json" and path.stem.endswith(("_result", "_results"))
)
saved_results = {}

for path in result_files:
    with path.open(encoding="utf-8") as f:
        saved_results[path.stem] = json.load(f)

print(f"Loaded {len(saved_results)} result file(s):")
for name in saved_results:
    print(f"- {name}.json")


Loaded 2 result file(s):
- hello_result.json
- matmul_result.json


## Compare result files

In [7]:
summary_rows = []

for result_name, data in saved_results.items():
    for target_name, target in data.items():
        if "return_code" in target:
            summary_rows.append(
                {
                    "result_file": result_name,
                    "target": target_name,
                    "kind": "subprocess",
                    "time_s": target.get("execution_time_s"),
                    "status": target.get("return_code"),
                    "checksum": target.get("checksum"),
                }
            )
            continue

        batches = target.get("batches")
        if batches:
            for batch in batches:
                summary_rows.append(
                    {
                        "result_file": result_name,
                        "target": target_name,
                        "kind": f"batch={batch.get('batch')}",
                        "time_s": batch.get("execution_time"),
                        "status": "ok",
                        "checksum": batch.get("checksum"),
                    }
                )
        else:
            summary_rows.append(
                {
                    "result_file": result_name,
                    "target": target_name,
                    "kind": "total",
                    "time_s": target.get("execution_time"),
                    "status": "ok",
                    "checksum": target.get("checksum"),
                }
            )

summary_rows


[{'result_file': 'hello_result',
  'target': 'hello',
  'kind': 'subprocess',
  'time_s': 0.08566659991629422,
  'status': 0,
  'checksum': None},
 {'result_file': 'matmul_result',
  'target': 'matmul',
  'kind': 'subprocess',
  'time_s': 0.172640300123021,
  'status': 0,
  'checksum': None}]

## Compare matrix multiplication batches

In [8]:
matmul_rows = [row for row in summary_rows if row["kind"].startswith("batch=")]

if not matmul_rows:
    print("No batch results found in results/*.json")
else:
    by_batch = {}
    for row in matmul_rows:
        batch = row["kind"].split("=", 1)[1]
        by_batch.setdefault(batch, []).append(row)

    comparison_rows = []
    for batch, rows in sorted(by_batch.items(), key=lambda item: int(item[0])):
        fastest = min(row["time_s"] for row in rows if row["time_s"] is not None)
        checksums = {row["checksum"] for row in rows if row["checksum"] is not None}
        for row in rows:
            speedup = fastest / row["time_s"] if row["time_s"] else None
            comparison_rows.append(
                {
                    "batch": int(batch),
                    "result_file": row["result_file"],
                    "target": row["target"],
                    "time_s": row["time_s"],
                    "speed_vs_fastest": speedup,
                    "checksum": row["checksum"],
                    "checksum_match": len(checksums) <= 1,
                }
            )

    comparison_rows


No batch results found in results/*.json
